In [ ]:
import urllib.request
import certifi
import ssl
import regex as re

In [ ]:
context = ssl.create_default_context(
    cafile=certifi.where()
)
texts = []
urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",
    "https://www.gutenberg.org/files/1342/1342-0.txt",
    "https://www.gutenberg.org/files/84/84-0.txt",
    "https://www.gutenberg.org/files/1661/1661-0.txt",
]

for url in urls:
    response = urllib.request.urlopen(
        url,
        context=context,
        timeout=30
    )
    text = response.read().decode("utf-8-sig")
    texts.append(text)
training_text = "\n\n".join(texts) + "<|endoftext|>"
print("Total characters:", len(training_text))
print("Total UTF-8 bytes:", len(training_text.encode("utf-8")))

In [ ]:
# print(sorted(stats))
# print(sorted(((v,k) for k,v in stats.items()),reverse=True))

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids,ids[1:]):
        counts[pair] = counts.get(pair,0)+1
    top_pair =max(counts,key=counts.get)
    return counts, top_pair

In [ ]:
def merge(ids,pair,idx):
    new_id = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_id.append(idx)
            i +=2
        else:
            new_id.append(ids[i])
            i +=1
    return new_id

In [ ]:
vocab_size = 270
current_size = 256
num_merges = vocab_size-current_size

pattern = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

special_tokens = {
    "<|endoftext|>": 270,
    '<|endofprompt|>':271,
    "<|pad|>": 272,
    "<|startoftext|>": 273,
    
}

special_pattern = re.compile(
    "(" + "|".join(re.escape(token) for token in special_tokens) + ")"
)

def pre_tokenize(text):
    parts = special_pattern.split(text)
    chunks = []
    for part in parts:
        if part in special_tokens:
            chunks.append(part)
        else:
            chunks.extend(pattern.findall(part))
    return chunks

In [ ]:
text = "Hello world!<|endoftext|>How are you?"

print(pre_tokenize(text))

In [ ]:
ids_gpt2 = []
chunks = pre_tokenize(training_text)
for chunk in chunks:
    if chunk in special_tokens:
        ids_gpt2.append([special_tokens[chunk]])
        continue
    ids_gpt2.append(list(chunk.encode("utf-8")))
gpt_merges = {}
for i in range(num_merges):
    stats = {}
    for piece in ids_gpt2:
        if len(piece)<2:
            continue
        piece_stats,_ = get_stats(piece)
        for pair, count in piece_stats.items():
            stats[pair] = stats.get(pair,0) + count
    if not stats:
        break
    top_pair = max(stats,key=stats.get)
    idx = 256 + i
    print(f"merging {top_pair} into new token {idx}")
    ids_gpt2 = [merge(piece, top_pair, idx) for piece in ids_gpt2]
    gpt_merges[top_pair] = idx

In [ ]:
print("token lenght",len(training_text))
print("ids length:", sum(len(piece) for piece in ids_gpt2))
print(f"compression ratio: {len(training_text) / sum(len(piece) for piece in ids_gpt2):.2f}X")

In [ ]:
gpt2_vocab  = {idx: bytes([idx]) for idx in range(256)}
for (p0,p1), idx in gpt_merges.items():
    gpt2_vocab[idx] = gpt2_vocab[p0] + gpt2_vocab[p1]
id_to_special = {
    idx: token for token, idx in special_tokens.items()
}
def gpt2_decode(ids):
    result = bytearray()
    for idx in ids:
        if idx in id_to_special:
            result.extend(id_to_special[idx].encode("utf-8"))
        else:
            result.extend(gpt2_vocab[idx])

    return result.decode("utf-8", errors="replace")

In [ ]:
def gpt2_encode(text,allowed_special=False):
    chunks = pre_tokenize(text)
    all_ids = []
    for chunk in chunks:
        if chunk in special_tokens and allowed_special:
            all_ids.append(special_tokens[chunk])
            continue
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            stats,_ = get_stats(ids)
            pair = min(stats,key=lambda p:gpt_merges.get(p,float("inf")))
            if pair not in gpt_merges:
                break 
            idx = gpt_merges[pair]
            ids = merge(ids,pair,idx)
        all_ids.extend(ids)
    return all_ids

In [ ]:
for text in [
    "Hello world! I'm learning GPT-2. <|endofprompt|>",
    "ہم کو تم سے پیار ہے",
    "زه له تاسره مینه لرم",
    "Hello 👋 World 🌍",
]:
    ids = gpt2_encode(text,allowed_special=True)
    restored = gpt2_decode(ids)
    print("Text:", text)
    print("IDs:", ids)
    print("Decoded correctly:", restored == text)
    assert restored == text
    print()

example = "Hello world! سلام دنیا"
print("Regex BPE token count:   ", len(gpt2_encode(example)))

In [ ]:
gpt2_encode("hi<|endoftext|>", allowed_special=True)   # one special ID at the end

In [ ]:
text = "hi<|endoftext|>"
gpt2_encode(text, allowed_special=False)

In [ ]:
text2 = gpt2_decode(gpt2_encode(text, allowed_special=False))
print(text == text2)